In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
import re
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================================
# TEXT FEATURE ENGINEERING
# ============================================================================
class TextFeatureExtractor:
    """Extract rich text features from catalog content"""
    
    def __init__(self):
        self.price_patterns = [
            r'\$\s*\d+(?:,\d{3})*(?:\.\d{2})?',
            r'\d+(?:,\d{3})*(?:\.\d{2})?\s*(?:dollars?|usd)',
            r'price[:\s]+\$?\d+',
        ]
        
    def extract_features(self, text):
        """Extract comprehensive text features"""
        text_lower = text.lower()
        
        features = {
            # Length features
            'char_count': len(text),
            'word_count': len(text.split()),
            'unique_word_count': len(set(text.lower().split())),
            'avg_word_length': np.mean([len(w) for w in text.split()]) if text.split() else 0,
            
            # Structural features
            'sentence_count': len(re.findall(r'[.!?]+', text)),
            'comma_count': text.count(','),
            'exclamation_count': text.count('!'),
            'question_count': text.count('?'),
            'uppercase_ratio': sum(1 for c in text if c.isupper()) / (len(text) + 1),
            'digit_ratio': sum(1 for c in text if c.isdigit()) / (len(text) + 1),
            
            # Product-specific keywords
            'has_warranty': int(any(w in text_lower for w in ['warranty', 'guarantee'])),
            'has_dimensions': int(any(w in text_lower for w in ['dimension', 'size', 'inches', 'cm', 'mm'])),
            'has_weight': int(any(w in text_lower for w in ['weight', 'pounds', 'lbs', 'kg', 'grams', 'oz'])),
            'has_material': int(any(w in text_lower for w in ['material', 'steel', 'plastic', 'wood', 'metal', 'cotton', 'leather'])),
            'has_color': int(any(w in text_lower for w in ['color', 'black', 'white', 'red', 'blue', 'green', 'silver', 'gold'])),
            'has_brand': int(any(w in text_lower for w in ['brand', 'manufacturer', 'made by'])),
            
            # Quality indicators
            'has_premium': int(any(w in text_lower for w in ['premium', 'luxury', 'professional', 'deluxe', 'pro'])),
            'has_budget': int(any(w in text_lower for w in ['budget', 'economy', 'basic', 'standard', 'affordable'])),
            'has_new': int('new' in text_lower),
            'has_pack': int(any(w in text_lower for w in ['pack', 'set', 'bundle', 'kit'])),
            
            # Numeric mentions
            'number_count': len(re.findall(r'\d+', text)),
            'has_percentage': int('%' in text),
            'has_measurement': int(bool(re.search(r'\d+\s*(?:inch|cm|mm|ft|meter|litre|ml|oz)', text_lower))),
        }
        
        return features

# ============================================================================
# DATASET CLASS
# ============================================================================
class ProductDataset(Dataset):
    def __init__(self, texts, text_features, labels=None, tokenizer=None, max_length=256):
        self.texts = texts
        self.text_features = text_features
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'text_features': torch.tensor(self.text_features[idx], dtype=torch.float32)
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
            
        return item

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================
class MultiHeadAttentionPooling(nn.Module):
    """Multi-head attention pooling for better sequence representation"""
    def __init__(self, hidden_size, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, hidden_size))
        
    def forward(self, hidden_states, attention_mask):
        batch_size = hidden_states.size(0)
        query = self.query.expand(batch_size, -1, -1)
        
        # Convert attention mask
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        attention_mask = (1.0 - attention_mask) * -10000.0
        
        output, _ = self.attention(query, hidden_states, hidden_states, 
                                  key_padding_mask=~attention_mask.squeeze().bool())
        return output.squeeze(1)

class AdvancedTextPriceModel(nn.Module):
    """Advanced model combining transformer encodings with text features"""
    
    def __init__(self, model_name='microsoft/deberta-v3-small', num_text_features=20, dropout=0.3):
        super().__init__()
        
        # Transformer backbone
        self.transformer = AutoModel.from_pretrained(model_name,use_fast=False)
        hidden_size = self.transformer.config.hidden_size
        
        # Freeze early layers for efficiency
        for param in self.transformer.embeddings.parameters():
            param.requires_grad = False
        
        # Advanced pooling strategies
        self.attention_pool = MultiHeadAttentionPooling(hidden_size, num_heads=8)
        self.mean_pool = True
        self.max_pool = True
        
        # Text feature processing
        self.text_feature_bn = nn.BatchNorm1d(num_text_features)
        self.text_feature_projection = nn.Sequential(
            nn.Linear(num_text_features, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU()
        )
        
        # Combine different pooling outputs
        combined_size = hidden_size * 3 + 128  # attention + mean + max + text features
        
        # Advanced prediction head
        self.predictor = nn.Sequential(
            nn.Linear(combined_size, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            
            nn.Linear(128, 1)
        )
        
    def forward(self, input_ids, attention_mask, text_features):
        # Get transformer outputs
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        
        # Multiple pooling strategies
        attention_pooled = self.attention_pool(hidden_states, attention_mask)
        
        # Mean pooling
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_hidden = torch.sum(hidden_states * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        
        # Max pooling
        max_pooled = torch.max(hidden_states * mask_expanded + (1 - mask_expanded) * -1e9, dim=1)[0]
        
        # Process text features
        text_feat_processed = self.text_feature_bn(text_features)
        text_feat_embedded = self.text_feature_projection(text_feat_processed)
        
        # Concatenate all representations
        combined = torch.cat([attention_pooled, mean_pooled, max_pooled, text_feat_embedded], dim=1)
        
        # Predict log price
        log_price = self.predictor(combined)
        return log_price.squeeze(-1)

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def smape_loss(pred, target):
    """Symmetric Mean Absolute Percentage Error loss"""
    denominator = (torch.abs(target) + torch.abs(pred)) / 2.0
    diff = torch.abs(pred - target) / torch.clamp(denominator, min=1e-8)
    return torch.mean(diff) * 100

def calculate_smape(y_true, y_pred):
    """Calculate SMAPE metric"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_pred - y_true) / np.maximum(denominator, 1e-8)
    return np.mean(diff) * 100

def train_epoch(model, dataloader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc='Training', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        text_features = batch['text_features'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(input_ids, attention_mask, text_features)
            loss = smape_loss(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        if scheduler is not None:
            scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate(model, dataloader, device):
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            text_features = batch['text_features'].to(device)
            labels = batch['labels'].to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(input_ids, attention_mask, text_features)
            
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Convert from log space back to original
    pred_prices = np.expm1(predictions)
    actual_prices = np.expm1(actuals)
    
    smape = calculate_smape(actual_prices, pred_prices)
    return smape, predictions

# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================
print("="*80)
print("LOADING DATA")
print("="*80)
train_df = pd.read_csv('/kaggle/input/ml-challenge-2025/train.csv')
test_df = pd.read_csv('/kaggle/input/ml-challenge-2025/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Train columns: {train_df.columns.tolist()}")
print(f"\nSample data:")
print(train_df.head(2))

# Extract text features
print("\n" + "="*80)
print("EXTRACTING TEXT FEATURES")
print("="*80)
feature_extractor = TextFeatureExtractor()

train_text_features = []
for text in tqdm(train_df['catalog_content'], desc='Train features'):
    features = feature_extractor.extract_features(str(text))
    train_text_features.append(list(features.values()))

test_text_features = []
for text in tqdm(test_df['catalog_content'], desc='Test features'):
    features = feature_extractor.extract_features(str(text))
    test_text_features.append(list(features.values()))

train_text_features = np.array(train_text_features)
test_text_features = np.array(test_text_features)

print(f"Text features shape: {train_text_features.shape}")

# Scale text features
scaler = RobustScaler()
train_text_features = scaler.fit_transform(train_text_features)
test_text_features = scaler.transform(test_text_features)

# Prepare target (log transform)
train_df['log_price'] = np.log1p(train_df['price'])

# Initialize tokenizer
print("\n" + "="*80)
print("LOADING TOKENIZER AND MODEL")
print("="*80)
model_name = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer loaded: {model_name}")

# Training configuration
CONFIG = {
    'n_folds': 5,
    'batch_size': 16,
    'epochs': 5,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'max_length': 256,
    'num_text_features': train_text_features.shape[1],
    'dropout': 0.3
}

print(f"Configuration: {CONFIG}")

# K-Fold Cross Validation
print("\n" + "="*80)
print("STARTING K-FOLD TRAINING")
print("="*80)

kfold = KFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=42)
oof_predictions = np.zeros(len(train_df))
test_predictions = np.zeros((len(test_df), CONFIG['n_folds']))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(train_df)):
    print(f"\n{'='*80}")
    print(f"FOLD {fold + 1}/{CONFIG['n_folds']}")
    print(f"{'='*80}")
    
    # Split data
    train_texts = train_df.iloc[train_idx]['catalog_content'].values
    val_texts = train_df.iloc[val_idx]['catalog_content'].values
    
    train_feats = train_text_features[train_idx]
    val_feats = train_text_features[val_idx]
    
    train_labels = train_df.iloc[train_idx]['log_price'].values
    val_labels = train_df.iloc[val_idx]['log_price'].values
    
    # Create datasets
    train_dataset = ProductDataset(train_texts, train_feats, train_labels, tokenizer, CONFIG['max_length'])
    val_dataset = ProductDataset(val_texts, val_feats, val_labels, tokenizer, CONFIG['max_length'])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'] * 2, shuffle=False, num_workers=2, pin_memory=True)
    
    # Initialize model
    model = AdvancedTextPriceModel(
        model_name=model_name,
        num_text_features=CONFIG['num_text_features'],
        dropout=CONFIG['dropout']
    ).to(device)
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    
    num_training_steps = len(train_loader) * CONFIG['epochs']
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CONFIG['learning_rate'],
        total_steps=num_training_steps,
        pct_start=0.1,
        anneal_strategy='cos'
    )
    
    scaler = torch.cuda.amp.GradScaler()
    
    best_smape = float('inf')
    
    # Training loop
    for epoch in range(CONFIG['epochs']):
        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, device)
        val_smape, val_preds = validate(model, val_loader, device)
        
        print(f"Train Loss: {train_loss:.4f} | Val SMAPE: {val_smape:.4f}")
        
        if val_smape < best_smape:
            best_smape = val_smape
            torch.save(model.state_dict(), f'best_model_fold{fold}.pt')
            print(f"✓ Best model saved! SMAPE: {best_smape:.4f}")
    
    fold_scores.append(best_smape)
    
    # Load best model for predictions
    model.load_state_dict(torch.load(f'best_model_fold{fold}.pt'))
    
    # Get OOF predictions
    _, oof_preds = validate(model, val_loader, device)
    oof_predictions[val_idx] = oof_preds
    
    # Get test predictions
    test_dataset = ProductDataset(test_df['catalog_content'].values, test_text_features, None, tokenizer, CONFIG['max_length'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'] * 2, shuffle=False, num_workers=2, pin_memory=True)
    
    model.eval()
    fold_test_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Test Prediction'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            text_features = batch['text_features'].to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(input_ids, attention_mask, text_features)
            fold_test_preds.extend(outputs.cpu().numpy())
    
    test_predictions[:, fold] = fold_test_preds
    
    print(f"\nFold {fold + 1} Best SMAPE: {best_smape:.4f}")

# Calculate overall OOF SMAPE
oof_prices = np.expm1(oof_predictions)
actual_prices = train_df['price'].values
overall_smape = calculate_smape(actual_prices, oof_prices)

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean Fold SMAPE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Overall OOF SMAPE: {overall_smape:.4f}")
print("="*80)

# Average test predictions
final_test_preds = np.mean(test_predictions, axis=1)
final_test_prices = np.expm1(final_test_preds)

# Create submission
submission = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_test_prices
})

submission.to_csv('submission.csv', index=False)
print("\n✓ Submission file created: submission.csv")
print(f"\nSubmission statistics:")
print(submission['price'].describe())
print(f"\nFirst 10 predictions:")
print(submission.head(10))